# GDG UAM ML Contest in Google Colab

Every submission is evaluated in Google Colab, so all teams are compared under the same conditions. Developing here is optional: any machine with Python 3.10+ works. This notebook gets the materials and the data, runs the baseline, and checks your own submission the way we will.

In [2]:
# the materials (safe to run again)
![ -d /content/ml-contest-materials ] || git clone -q https://github.com/GDG-UAM/ml-contest-materials.git /content/ml-contest-materials
%cd /content/ml-contest-materials

/content/ml-contest-materials


In [3]:
# training data (413 MB) and the demo test set (1.7 MB), from the latest release
!curl -sL --create-dirs -o data/train_100k.npz https://github.com/GDG-UAM/ml-contest-materials/releases/latest/download/train_100k.npz
!curl -sL --create-dirs -o data/test_demo.npz https://github.com/GDG-UAM/ml-contest-materials/releases/latest/download/test_demo.npz

In [4]:
# the baseline: train it, then evaluate it like the real thing, and on the demo test set
!python examples/baseline_train.py
!python harness.py evaluate --model examples/baseline_model.py
!python harness.py evaluate --model examples/baseline_model.py --source file:data/test_demo.npz

features: 101 per sample for 100000 samples (84 s)
validation accuracy: 0.821 on 20000 samples
saved /content/ml-contest-materials/examples/baseline_model.pkl
evaluating baseline_model (examples/baseline_model.py)
  [baseline_model] run 0  score=13.862  n=1857  acc=0.976  latency=32.3ms  end=time_budget  ok
  [baseline_model] run 1  score=13.642  n=1931  acc=0.965  latency=31.1ms  end=time_budget  ok
  [baseline_model] run 2  score=13.481  n=1897  acc=0.966  latency=31.6ms  end=time_budget  ok
rank  project          mean   best  worst  median  trimmed_mean  mean_minus_std  failed_runs  mean_accuracy_all  mean_n_scored  mean_latency_s
----------------------------------------------------------------------------------------------------------------------------------------------
   1  baseline_model  13.66  13.86  13.48   13.64         13.66           13.51            0             0.9688           1895         0.03166
evaluating baseline_model (examples/baseline_model.py)
  [baseline_model

In [5]:
from google.colab import userdata, _message
import os, subprocess, json

os.environ['GI'] = userdata.get('GI')
!git config --global user.email "christianmenhem23@example.com"
!git config --global user.name "Christian-m23"

In [ ]:

TOKEN = userdata.get("GI")
REPO  = "github.com/Christian-m23/ml-contest-gdguam.git"
DIR   = "/content/repo"
URL   = f"https://{TOKEN}@{REPO}"

# clone or update
if not os.path.isdir(DIR):
    subprocess.run(["git", "clone", URL, DIR], check=True)
else:
    subprocess.run(["git", "-C", DIR, "pull", "--rebase"], check=True)

# grab the CURRENT notebook (with all your edits) from Colab's memory
resp = _message.blocking_request("get_ipynb", request="")
nb = resp["ipynb"]
if isinstance(nb, str):
    nb = json.loads(nb)
with open(f"{DIR}/colab.ipynb", "w") as f:
    json.dump(nb, f, indent=1)

# push
subprocess.run(["git", "-C", DIR, "add", "-A"], check=True)
subprocess.run(["git", "-C", DIR, "commit", "-m", "sync from colab"], check=False)
subprocess.run(["git", "-C", DIR, "push", URL, "main"], check=True)
print("✅ pushed: https://github.com/Christian-m23/ml-contest-gdguam")

In [ ]:
# BLOCK 1 — Load data and print schema, side by side
# Loads train and demo, prints shape/dtype/writable/class counts/NaN rate,
# and (via the baseline's own isinstance check) which meta fields survive.

import numpy as np
from competition_api import load_dataset

train = load_dataset("data/train_100k.npz")
test  = load_dataset("data/test_demo.npz")

def baseline_keeps_key(arr, k=500):
    """True iff baseline meta_features() keeps this field (sampled check)."""
    idx = np.linspace(0, len(arr) - 1, min(k, len(arr))).astype(int)
    for i in idx:
        v = arr[i]
        v = v.item() if hasattr(v, "item") else v
        if isinstance(v, bool) or not isinstance(v, (int, float)):
            return False
    return True

def row(label, tr, te, wl=24, wv=34):
    print(f"  {label:<{wl}} {str(tr):<{wv}} {str(te):<{wv}}")

print("=" * 96)
print("SCHEMA — TRAIN vs DEMO TEST")
print("=" * 96)
print(f"  {'':<24} {'TRAIN':<34} {'DEMO TEST':<34}")
print(f"  {'-'*24} {'-'*34} {'-'*34}")

row("n samples",            len(train),                      len(test))
row("series shape",         str(train.series.shape),         str(test.series.shape))
row("series dtype",         str(train.series.dtype),         str(test.series.dtype))
row("series writable",      train.series.flags.writeable,    test.series.flags.writeable)
row("series mem (MB)",      f"{train.series.nbytes/1e6:.1f}",f"{test.series.nbytes/1e6:.1f}")
row("labels shape",         str(train.labels.shape),         str(test.labels.shape))
row("labels dtype",         str(train.labels.dtype),         str(test.labels.dtype))
row("labels range",         f"[{train.labels.min()}, {train.labels.max()}]",
                            f"[{test.labels.min()}, {test.labels.max()}]")
row("ds[0] type",           type(train[0]).__name__,         type(test[0]).__name__)
row("ds[0].series shape",   str(train[0].series.shape),      str(test[0].series.shape))
row("ds[0].series writable",train[0].series.flags.writeable, test[0].series.flags.writeable)
row("NaN rate overall",     f"{np.isnan(train.series).mean()*100:.3f}%",
                            f"{np.isnan(test.series).mean()*100:.3f}%")

print("\n  CLASS COUNTS")
tr_c = np.bincount(train.labels[train.labels >= 0], minlength=8)
te_c = np.bincount(test.labels[test.labels >= 0],   minlength=8)
for k in range(8):
    print(f"    class {k}: train={tr_c[k]:>7}   demo={te_c[k]:>5}")
print(f"    total    : train={tr_c.sum():>7}   demo={te_c.sum():>5}")

print("\n  META FIELDS")
for key in sorted(train.meta):
    tr_arr, te_arr = train.meta[key], test.meta[key]
    tr_u, te_u = np.unique(tr_arr), np.unique(te_arr)
    print(f"    {key!r}")
    print(f"      train: dtype={tr_arr.dtype}  n_unique={len(tr_u)}  first10={tr_u[:10].tolist()}")
    print(f"      demo : dtype={te_arr.dtype}  n_unique={len(te_u)}  first10={te_u[:10].tolist()}")
    print(f"      baseline meta_features() keeps:  train={baseline_keeps_key(tr_arr)}  "
          f"demo={baseline_keeps_key(te_arr)}")

print("\n  WHAT baseline meta_features() ACTUALLY RETURNS (sample 0)")
for name, ds in [("TRAIN", train), ("DEMO", test)]:
    m = ds[0].meta
    kept = [k for k, v in sorted(m.items())
            if not isinstance(v, bool) and isinstance(v, (int, float))]
    dropped = [k for k, v in sorted(m.items())
               if isinstance(v, bool) or not isinstance(v, (int, float))]
    print(f"    {name}: meta={m}")
    print(f"      kept (order): {kept}   len={len(kept)}")
    print(f"      dropped     : {dropped}")
    print(f"      total vector length = 92 + 6 + {len(kept)} + 1 = {92 + 6 + len(kept) + 1}")

In [ ]:
# BLOCK 9a — Spike flag via robust z-score.
# Per sample, per channel: z = max|x - median| / (1.4826 * MAD)
# Flag the sample if any channel's z > 8.
# MAD == 0: 0/0 -> 0 (not flagged); x/0 with x>0 -> inf (flagged).

import warnings

def flag_spikes_batch(series_batch, z_thresh=8.0, k=1.4826):
    """series_batch: (B, 4, L) float32. Returns (flagged: (B,) bool, z: (B, 4))."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        med = np.nanmedian(series_batch, axis=2)            # (B, 4)
        dev = np.abs(series_batch - med[:, :, None])        # (B, 4, L)
        mad = np.nanmedian(dev, axis=2)                     # (B, 4)
        max_dev = np.nanmax(dev, axis=2)                    # (B, 4)
        with np.errstate(divide="ignore", invalid="ignore"):
            z = max_dev / (k * mad)                         # (B, 4)
        z = np.where(np.isnan(z), 0.0, z)                   # 0/0 -> 0
        flagged = np.any(z > z_thresh, axis=1)              # (B,)
    return flagged, z

def measure_flags(series, batch=10000, z_thresh=8.0):
    n = len(series)
    flags = np.zeros(n, dtype=bool)
    z_max = np.zeros(n, dtype=np.float32)
    for start in range(0, n, batch):
        end = min(start + batch, n)
        f, z = flag_spikes_batch(series[start:end], z_thresh=z_thresh)
        flags[start:end] = f
        z_max[start:end] = z.max(axis=1)
    return flags, z_max

print("Computing spike flags on TRAIN (100k)...")
train_flags, train_z = measure_flags(train.series)
print("Computing spike flags on DEMO (400)...")
demo_flags, demo_z = measure_flags(test.series)

print("\n" + "=" * 72)
print("SPIKE FLAG — fraction flagged (robust z > 8 on any channel)")
print("=" * 72)
print(f"  TRAIN : {int(train_flags.sum()):>6} / {len(train):>6}  = {train_flags.mean()*100:5.3f}%")
print(f"  DEMO  : {int(demo_flags.sum()):>6} / {len(test):>6}  = {demo_flags.mean()*100:5.3f}%")

print(f"\n  TRAIN z distribution (max over channels): "
      f"p50={np.percentile(train_z, 50):.2f}  "
      f"p95={np.percentile(train_z, 95):.2f}  "
      f"p99={np.percentile(train_z, 99):.2f}  "
      f"p99.9={np.percentile(train_z, 99.9):.2f}  "
      f"max={train_z.max():.2f}")
print(f"  DEMO  z distribution (max over channels): "
      f"p50={np.percentile(demo_z, 50):.2f}  "
      f"p95={np.percentile(demo_z, 95):.2f}  "
      f"p99={np.percentile(demo_z, 99):.2f}  "
      f"p99.9={np.percentile(demo_z, 99.9):.2f}  "
      f"max={demo_z.max():.2f}")

print("\n  DEMO — flagged count per class")
for k in range(8):
    m = test.labels == k
    if m.sum() > 0:
        print(f"    class {k}: {int(demo_flags[m].sum()):>3} / {int(m.sum()):>3}")

# Auditing


In [ ]:
# BLOCK 9b — Run the baseline Model on all 400 demo samples.
# Report accuracy on flagged vs unflagged; list every flagged sample.

import importlib.util, sys
from pathlib import Path

if not Path("examples/baseline_model.pkl").exists():
    print("⚠️ examples/baseline_model.pkl not found. Run this first, then re-run this block:")
    print("   !python examples/baseline_train.py")
else:
    spec = importlib.util.spec_from_file_location(
        "baseline_model", "examples/baseline_model.py")
    mod = importlib.util.module_from_spec(spec)
    sys.modules["baseline_model"] = mod
    spec.loader.exec_module(mod)

    print("Loading baseline Model and predicting on 400 demo samples...")
    m = mod.Model()
    preds = np.empty(len(test), dtype=int)
    for i in range(len(test)):
        preds[i] = m.predict(test[i])
        if (i + 1) % 100 == 0:
            print(f"  ... {i + 1}/{len(test)} done")

    labels = test.labels
    flagged = demo_flags

    print("\n" + "=" * 72)
    print("BASELINE accuracy on DEMO — spike-flagged vs unflagged")
    print("=" * 72)
    for name, mask in [("UNFLAGGED", ~flagged), ("FLAGGED", flagged)]:
        n = int(mask.sum())
        if n == 0:
            print(f"  {name:<10} n=0")
            continue
        acc = (preds[mask] == labels[mask]).mean()
        print(f"  {name:<10} n={n:>3}  accuracy={acc:.3f}")

    if flagged.any() and (~flagged).any():
        acc_f = (preds[flagged]  == labels[flagged]).mean()
        acc_u = (preds[~flagged] == labels[~flagged]).mean()
        print(f"\n  gap: unflagged - flagged = {acc_u - acc_f:+.3f}")

    if flagged.any():
        print("\n  Individual flagged demo samples:")
        print(f"    {'i':>4} {'true':>5} {'pred':>5} {'ok':>3} {'z_max':>8} {'level':>8}")
        for i in np.where(flagged)[0]:
            ok = "Y" if int(preds[i]) == int(labels[i]) else "N"
            print(f"    {int(i):>4} {int(labels[i]):>5} {int(preds[i]):>5} "
                  f"{ok:>3} {float(demo_z[i]):>8.2f} {float(test.meta['level'][i]):>8.3f}")

In [ ]:
# BLOCK 2 — Label balance and meta distributions, side by side

def label_balance(train, test):
    print("=" * 76)
    print("LABEL BALANCE — TRAIN vs DEMO")
    print("=" * 76)
    tr = np.bincount(train.labels[train.labels >= 0], minlength=8)
    te = np.bincount(test.labels[test.labels >= 0],   minlength=8)
    print(f"  {'cls':>4} {'train n':>10} {'train%':>8} {'demo n':>8} {'demo%':>8}")
    for k in range(8):
        print(f"  {k:>4} {tr[k]:>10} {tr[k]/tr.sum()*100:>7.2f}% {te[k]:>8} {te[k]/te.sum()*100:>7.2f}%")
    print(f"  {'sum':>4} {tr.sum():>10} {'100.00%':>8} {te.sum():>8} {'100.00%':>8}")
    print(f"  max/min ratio: train={tr.max()/tr.min():.2f}x   demo={te.max()/te.min():.2f}x")

def meta_distributions(train, test):
    print("\n" + "=" * 76)
    print("META DISTRIBUTIONS — TRAIN vs DEMO")
    print("=" * 76)
    for key in ["sample_rate", "group"]:
        print(f"\n  [{key}] (discrete)")
        tr, te = train.meta[key], test.meta[key]
        vals = sorted(set(np.unique(tr).tolist()) | set(np.unique(te).tolist()))
        print(f"    {'value':>10} {'train n':>10} {'train%':>8} {'demo n':>8} {'demo%':>8}")
        for v in vals:
            trn, ten = int((tr == v).sum()), int((te == v).sum())
            print(f"    {str(v):>10} {trn:>10} {trn/len(tr)*100:>7.2f}% {ten:>8} {ten/len(te)*100:>7.2f}%")

    print(f"\n  [level] (continuous)")
    trl, tel = train.meta["level"].astype(float), test.meta["level"].astype(float)
    qs = [0, 1, 5, 25, 50, 75, 95, 99, 100]
    print(f"    {'stat':>8} {'train':>14} {'demo':>14}")
    for q in qs:
        print(f"    p{q:<7} {np.percentile(trl, q):>14.4f} {np.percentile(tel, q):>14.4f}")
    print(f"    {'mean':>8} {trl.mean():>14.4f} {tel.mean():>14.4f}")
    print(f"    {'std':>8} {trl.std():>14.4f} {tel.std():>14.4f}")

label_balance(train, test)
meta_distributions(train, test)

In [ ]:
# BLOCK 3 (rewritten) — Per-channel stats, train vs demo.
# Adds median and MAD (median absolute deviation) alongside mean/std.
# MAD = median(|x - median(x)|); robust to outliers, comparable in scale to std for Gaussian data.

def _mad(x):
    x = x[~np.isnan(x)]
    if len(x) == 0:
        return np.nan
    return np.median(np.abs(x - np.median(x)))

def channel_stats(train, test):
    print("=" * 110)
    print("PER-CHANNEL STATS — TRAIN vs DEMO (NaN-aware; median and MAD added)")
    print("=" * 110)
    hdr = (f"  {'ch':>3} {'src':>6} {'mean':>10} {'std':>10} "
           f"{'median':>10} {'MAD':>10} {'min':>10} {'p1':>10} {'p99':>10} {'max':>10}")
    print(hdr)
    for ch in range(4):
        for name, ds in [("train", train), ("demo", test)]:
            x = ds.series[:, ch, :]
            print(f"  {ch:>3} {name:>6} "
                  f"{np.nanmean(x):>10.4f} {np.nanstd(x):>10.4f} "
                  f"{np.nanmedian(x):>10.4f} {_mad(x):>10.4f} "
                  f"{np.nanmin(x):>10.4f} {np.nanpercentile(x, 1):>10.4f} "
                  f"{np.nanpercentile(x, 99):>10.4f} {np.nanmax(x):>10.4f}")
    print("\n  Note: MAD compares to std like-for-like on Gaussian data "
          "(MAD ≈ 0.6745·std). A large std/MAD ratio flags outlier dominance.")

channel_stats(train, test)

In [ ]:
# BLOCK 4 — NaN fraction and pattern.
# Prints overall rate, per-channel, per-sample stats, and rate by time position.
# Plots: (a) NaN rate per timestep, train vs demo; (b) histogram of per-sample NaN count.

import matplotlib.pyplot as plt

def nan_pattern(train, test):
    print("=" * 76)
    print("NaN PATTERN — TRAIN vs DEMO")
    print("=" * 76)
    for name, ds in [("TRAIN", train), ("DEMO", test)]:
        m = np.isnan(ds.series)
        per_sample = m.reshape(len(ds), -1).sum(1)
        per_time = m.mean(axis=(0, 1)) * 100
        print(f"\n  [{name}]")
        print(f"    overall rate         : {m.mean()*100:.4f}%")
        print(f"    per-channel rate     : " + "  ".join(
            f"ch{c}={m[:, c, :].mean()*100:.4f}%" for c in range(4)))
        print(f"    per-sample NaN count : mean={per_sample.mean():.1f}  "
              f"median={np.median(per_sample):.0f}  max={per_sample.max()}")
        print(f"    samples with 0 NaN   : {(per_sample == 0).mean()*100:.2f}%")
        print(f"    samples with >100 NaN: {(per_sample > 100).mean()*100:.2f}%")
        print(f"    by time: first10={per_time[:10].mean():.3f}%   "
              f"mid(78:178)={per_time[78:178].mean():.3f}%   "
              f"last10={per_time[-10:].mean():.3f}%")

    fig, axes = plt.subplots(1, 2, figsize=(14, 3.5), sharey=True)
    for ax, (name, ds) in zip(axes, [("train", train), ("demo", test)]):
        pt = np.isnan(ds.series).mean(axis=(0, 1)) * 100
        ax.plot(pt)
        ax.set_title(f"NaN rate per timestep — {name} (n={len(ds)})")
        ax.set_xlabel("timestep"); ax.set_ylabel("% NaN"); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    fig, ax = plt.subplots(figsize=(10, 3.5))
    for name, ds, c in [("train", train, "C0"), ("demo", test, "C1")]:
        ps = np.isnan(ds.series).reshape(len(ds), -1).sum(1)
        ax.hist(ps, bins=50, alpha=0.55, label=name, color=c, density=True)
    ax.set_xlabel("NaN count per sample (out of 1024)")
    ax.set_ylabel("density"); ax.set_title("Per-sample NaN count")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

nan_pattern(train, test)

In [ ]:
# BLOCK 5 (rewritten) — Median power spectrum per class.
# Row 1: full train (n≈10k–16k per class).
# Row 2: stratified 400 from train (50 per class, = demo's per-class n) — noise floor control.
# Row 3: demo (50 per class).
# Median across samples instead of mean, so a few extreme samples can't dominate.
# NaN filled with 0 before FFT (cheapest option; fine for a per-class overview).

def median_power_spectrum(train, test, seed=0):
    rng = np.random.default_rng(seed)

    # stratified 400 from train: 50 per class
    strat_idx = []
    for k in range(8):
        idx = np.where(train.labels == k)[0]
        picked = rng.choice(idx, size=min(50, len(idx)), replace=False)
        strat_idx.append(picked)
    strat_idx = np.concatenate(strat_idx)

    sources = [
        ("train (full)",     train.series, train.labels),
        ("train (strat 400)", train.series[strat_idx], train.labels[strat_idx]),
        ("demo",             test.series,  test.labels),
    ]

    fig, axes = plt.subplots(3, 4, figsize=(20, 10), sharex=True)
    for row, (src_name, s, labels) in enumerate(sources):
        for ch in range(4):
            ax = axes[row, ch]
            for k in range(8):
                m = labels == k
                if m.sum() == 0:
                    continue
                x = np.where(np.isnan(s[m, ch, :]), 0.0, s[m, ch, :])
                psd = np.abs(np.fft.rfft(x, axis=1)) ** 2   # (n_k, n_freq)
                ax.plot(np.arange(psd.shape[1]), np.median(psd, axis=0),
                        label=f"class {k}", alpha=0.8)
            ax.set_yscale("log")
            ax.set_title(f"{src_name} — ch{ch}  (median PSD)")
            ax.set_xlabel("freq bin")
            ax.grid(alpha=0.3)
            if ch == 0:
                ax.set_ylabel(f"{src_name}\nmedian |rfft|²")
            if row == 0 and ch == 3:
                ax.legend(fontsize=7, ncol=2, loc="upper right")
    plt.tight_layout(); plt.show()

median_power_spectrum(train, test, seed=0)

In [ ]:
# BLOCK 6 — Eyeball plots: 3 random samples per class per channel per source.
# One figure per source. 8 rows (classes) x 4 columns (channels). 3 traces per subplot.

def plot_signals(ds, src_name, n_per_class=3, seed=0):
    rng = np.random.default_rng(seed)
    fig, axes = plt.subplots(8, 4, figsize=(20, 16), sharex=True)
    s, labels = ds.series, ds.labels
    for k in range(8):
        idx = np.where(labels == k)[0]
        if len(idx) == 0:
            for ch in range(4):
                axes[k, ch].set_title(f"class {k} — no samples")
            continue
        picked = rng.choice(idx, size=min(n_per_class, len(idx)), replace=False)
        for ch in range(4):
            ax = axes[k, ch]
            for i in picked:
                ax.plot(s[i, ch, :], alpha=0.75)
            ax.set_title(f"class {k} — ch{ch}  (n={len(idx)})", fontsize=9)
            ax.grid(alpha=0.2)
            if ch == 0: ax.set_ylabel(f"cls {k}")
    fig.suptitle(f"{src_name}: 3 random samples per class per channel", y=1.005)
    plt.tight_layout(); plt.show()

plot_signals(train, "TRAIN", n_per_class=3, seed=0)
plot_signals(test,  "DEMO TEST", n_per_class=3, seed=0)

In [ ]:
# BAND COMPARISON TEST (rewritten) — additive-noise hypothesis
# Tests: "the extra variation in demo is additive noise, same variance for all classes"
#
# Three refinements vs the original:
#   (1) LF band (bins 1-29) alongside HF (bins 100-128).
#       White noise → LF diff/N ≈ HF diff/N.
#       HF-specific noise → HF >> LF.   LF-specific → LF >> HF.
#   (2) Bootstrap 95% CI on each class's diff (demo n≈46; train n≈10k, treated as fixed).
#   (3) Bootstrap null distribution of the cross-class CV.
#       p(null CV ≥ observed CV) tests "is observed spread beyond sampling noise?"
#       Small p → reject white noise (class-dependent). High p → consistent with white noise.

BAND_LF = slice(1, 30)
BAND_HF = slice(100, 129)
N_TRAIN = 20_000
N = 256            # rfft length; for white noise of var σ², E[|rfft|²] = N·σ²
N_BOOT = 500

def fill_nan(x):
    out = np.array(x, dtype=np.float64)
    t = np.arange(out.shape[1])
    for ch in out:
        bad = np.isnan(ch)
        if bad.all():
            ch[:] = 0.0
        elif bad.any():
            ch[bad] = np.interp(t[bad], t[~bad], ch[~bad])
    return out

def band_power(series, band):
    """(B, 4, 256) -> (B, 4): mean |rfft|^2 over band, NaNs interpolated."""
    out = np.empty((len(series), series.shape[1]))
    for i in range(len(series)):
        out[i] = (np.abs(np.fft.rfft(fill_nan(series[i]), axis=1))[:, band] ** 2).mean(axis=1)
    return out

# ---- select rows: unflagged only, on both sides
rng = np.random.default_rng(0)
idx = rng.permutation(len(train))[:N_TRAIN]
idx = np.sort(idx[~train_flags[idx]])
keep = ~demo_flags
y_tr = train.labels[idx]
y_te = test.labels[keep]

print(f"train rows used: {len(idx)}   demo unflagged rows used: {keep.sum()}")
print("Computing band power (LF and HF)...")
P_LF_tr = band_power(train.series[idx], BAND_LF)
P_LF_te = band_power(test.series[keep], BAND_LF)
P_HF_tr = band_power(train.series[idx], BAND_HF)
P_HF_te = band_power(test.series[keep], BAND_HF)

# ---- per (band, channel) bootstrap analysis
def band_stats(P_tr, P_te, y_tr, y_te, ch, n_boot=N_BOOT, seed=0):
    rng_b = np.random.default_rng(seed)
    per_class_demo = [P_te[y_te == k, ch] for k in range(8)]
    train_med = np.array([np.median(P_tr[y_tr == k, ch]) for k in range(8)])
    demo_med  = np.array([np.median(per_class_demo[k]) for k in range(8)])
    diff      = demo_med - train_med

    # bootstrap: resample demo per class (train median treated as fixed)
    boot_diff = np.zeros((n_boot, 8))
    for b in range(n_boot):
        for k in range(8):
            s = rng_b.choice(per_class_demo[k], size=len(per_class_demo[k]), replace=True)
            boot_diff[b, k] = np.median(s) - train_med[k]

    ci_lo = np.percentile(boot_diff,  2.5, axis=0)
    ci_hi = np.percentile(boot_diff, 97.5, axis=0)

    null_cv = boot_diff.std(axis=1) / np.abs(boot_diff.mean(axis=1))
    obs_cv  = diff.std() / abs(diff.mean()) if diff.mean() != 0 else np.nan
    p_ge    = float((null_cv >= obs_cv).mean())    # one-sided: "more spread than sampling noise"
    return dict(train_med=train_med, demo_med=demo_med, diff=diff,
                ci_lo=ci_lo, ci_hi=ci_hi, obs_cv=obs_cv,
                null_median=float(np.median(null_cv)),
                null_p975=float(np.percentile(null_cv, 97.5)),
                p_ge=p_ge)

results = {}
for band_name, P_tr, P_te in [("LF", P_LF_tr, P_LF_te), ("HF", P_HF_tr, P_HF_te)]:
    for ch in range(4):
        seed = (0 if band_name == "LF" else 1000) + ch
        s = band_stats(P_tr, P_te, y_tr, y_te, ch, seed=seed)
        results[(band_name, ch)] = s

        print("\n" + "=" * 118)
        print(f"BAND={band_name}   CH{ch}   diff = demo_median − train_median  "
              f"(95% CI from {N_BOOT} bootstrap resamples of demo)")
        print("=" * 118)
        print(f"  {'cls':>3} {'n_demo':>6} {'train_med':>10} {'demo_med':>10} {'diff':>10} "
              f"{'diff 95% CI':>22} {'diff/N':>9}")
        for k in range(8):
            ci = f"[{s['ci_lo'][k]:+.2f}, {s['ci_hi'][k]:+.2f}]"
            n_demo = int((y_te == k).sum())
            print(f"  {k:>3} {n_demo:>6} {s['train_med'][k]:>10.2f} {s['demo_med'][k]:>10.2f} "
                  f"{s['diff'][k]:>10.2f} {ci:>22} {s['diff'][k]/N:>9.4f}")
        print(f"\n  Observed CV across classes     : {s['obs_cv']:.3f}")
        print(f"  Null CV (bootstrap): median    : {s['null_median']:.3f}   p97.5: {s['null_p975']:.3f}")
        print(f"  p(null CV ≥ observed CV)       : {s['p_ge']:.3f}   "
              f"(small → spread > sampling noise → reject white noise)")

# ---- LF vs HF summary (the broadband test)
print("\n" + "=" * 86)
print("BROADBAND TEST — mean(diff/N) by band and channel")
print("=" * 86)
print(f"  {'ch':>3} {'LF mean diff/N':>18} {'HF mean diff/N':>18} {'LF/HF':>10}")
for ch in range(4):
    lf = results[("LF", ch)]["diff"].mean() / N
    hf = results[("HF", ch)]["diff"].mean() / N
    ratio_str = f"{lf/hf:.2f}" if hf != 0 else ("inf" if lf > 0 else "n/a")
    print(f"  {ch:>3} {lf:>18.4f} {hf:>18.4f} {ratio_str:>10}")

print("\n  Interpretation:")
print("    LF ≈ HF, both > 0          → white noise (broadband)")
print("    HF >> LF, HF > 0           → high-frequency-specific noise")
print("    LF >> HF, LF > 0           → low-frequency-specific (drift / detrending)")
print("    LF ≈ HF ≈ 0                → no added noise in power; std gap is spike- or class-driven")

# Classifier on named measurements


In [ ]:
# BLOCK 11 — Adversarial validation on named measurements (train vs demo)
# Needs: train, test, train_flags, demo_flags (Block 9a). Runtime: several minutes.
import warnings
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

C = 4
LF, MID, HF = slice(1, 9), slice(20, 61), slice(100, 129)   # rfft bins
Z = 8.0                                                      # robust-z spike threshold

# ------------------------------------------------------------------ named measurements
def _longest_run(b):
    best = cur = 0
    for v in b:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best

def _interp(x):
    out = x.copy()
    t = np.arange(x.shape[1])
    for ch in out:
        bad = np.isnan(ch)
        if bad.all():
            ch[:] = 0.0
        elif bad.any():
            ch[bad] = np.interp(t[bad], t[~bad], ch[~bad])
    return out

def named_features(x, meta):
    x = np.asarray(x, dtype=np.float64)
    nan = np.isnan(x)
    f = {"nan_frac": nan.mean()}
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        med = np.nanmedian(x, axis=1)
        dev = np.abs(x - med[:, None])
        mad = np.nanmedian(dev, axis=1)
    med, mad = np.nan_to_num(med), np.nan_to_num(mad)
    sig = 1.4826 * mad + 1e-9
    rz = np.nan_to_num(dev / sig[:, None])                    # robust z of every point
    f["spike_n_pts"] = float((rz > Z).sum())
    for c in range(C):
        f[f"nan_gap_ch{c}"] = _longest_run(nan[c])
        f[f"spike_logz_ch{c}"] = np.log1p(rz[c].max())
        f[f"med_ch{c}"] = med[c]
        f[f"logmad_ch{c}"] = np.log(mad[c] + 1e-9)

    # spike-clipped, NaN-interpolated copy: spectral/shape features see no spikes
    clean = np.clip(_interp(x), (med - Z * sig)[:, None], (med + Z * sig)[:, None])
    P = np.abs(np.fft.rfft(clean, axis=1)) ** 2
    zc = (clean - clean.mean(1, keepdims=True)) / (clean.std(1, keepdims=True) + 1e-9)
    for c in range(C):
        f[f"lf_ch{c}"] = np.log(P[c, LF].mean() + 1e-9)
        f[f"mid_ch{c}"] = np.log(P[c, MID].mean() + 1e-9)
        f[f"hf_ch{c}"] = np.log(P[c, HF].mean() + 1e-9)
        f[f"skew_ch{c}"] = (zc[c] ** 3).mean()
        f[f"kurt_ch{c}"] = (zc[c] ** 4).mean() - 3.0
        f[f"ac1_ch{c}"] = (zc[c, 1:] * zc[c, :-1]).mean()
    with np.errstate(all="ignore"):
        cc = np.nan_to_num(np.corrcoef(clean))
    for a in range(C):
        for b in range(a + 1, C):
            f[f"xc_{a}{b}"] = cc[a, b]

    f["level"], f["sample_rate"] = float(meta["level"]), float(meta["sample_rate"])
    for g in "ABC":
        f[f"group_{g}"] = float(meta["group"] == g)

    # "rich" spectrum, used only for the completeness check: 32 log-bins per channel
    pr = P[:, 1:129].reshape(C, 32, 4).mean(axis=2)
    for c in range(C):
        for j in range(32):
            f[f"rich_ch{c}_b{j}"] = np.log(pr[c, j] + 1e-9)
    return f

def featurize(ds, idx):
    return pd.DataFrame([named_features(ds.series[i], ds.meta_at(i)) for i in idx], index=idx)

PREFIX = [("nan_", "nan"), ("spike_", "spike"), ("med_", "scale"), ("logmad_", "scale"),
          ("lf_", "lf"), ("mid_", "mid"), ("hf_", "hf"),
          ("skew_", "shape"), ("kurt_", "shape"), ("ac1_", "shape"),
          ("xc_", "xcorr"), ("rich_", "rich"),
          ("level", "meta"), ("sample_rate", "meta"), ("group_", "meta")]
group_of = lambda col: next(g for p, g in PREFIX if col.startswith(p))

# ------------------------------------------------------------------ data
POOL_PER_CLASS, N_DEMO_LIKE, N_TRAIN_SIDE = 300, 50, 250
rng = np.random.default_rng(0)
pool_idx = np.concatenate([rng.permutation(np.where(train.labels == k)[0])[:POOL_PER_CLASS]
                           for k in range(8)])
print("featurizing pool and demo ...")
F_pool = featurize(train, pool_idx)
F_demo = featurize(test, np.arange(len(test)))
y_pool, flag_pool = train.labels[pool_idx], train_flags[pool_idx]
NAMED = [c for c in F_pool.columns if group_of(c) != "rich"]
ALL = list(F_pool.columns)
GROUPS = sorted({group_of(c) for c in NAMED})
print(f"pool {F_pool.shape}, demo {F_demo.shape}, {len(NAMED)} named features in {len(GROUPS)} groups")

# ------------------------------------------------------------------ adversarial classifier
def make_clf():
    return HistGradientBoostingClassifier(max_iter=100, learning_rate=0.08, max_depth=3,
                                          min_samples_leaf=20, class_weight="balanced", random_state=0)

def adv_auc(A, B, cols, reps=3):
    """Out-of-fold AUC of 'row is from A (label 1) vs B (label 0)'. Also mean OOF score of A's rows."""
    X = np.vstack([A[cols].values, B[cols].values])
    y = np.r_[np.ones(len(A)), np.zeros(len(B))]
    aucs, oof = [], np.zeros(len(X))
    for r in range(reps):
        p = cross_val_predict(make_clf(), X, y, method="predict_proba",
                              cv=StratifiedKFold(5, shuffle=True, random_state=r))[:, 1]
        aucs.append(roc_auc_score(y, p))
        oof += p / reps
    return float(np.mean(aucs)), oof[:len(A)]

def draw(kind, k, peel):
    """control: pseudo-demo (50/class) vs train side (250/class), both from the train pool.
       real: the demo vs train side (250/class). Same sizes, same class balance."""
    r = np.random.default_rng(1000 * (kind == "control") + k)
    keep = ~flag_pool if peel else np.ones(len(y_pool), bool)
    a_rows, b_rows = [], []
    for c in range(8):
        rows = r.permutation(np.where((y_pool == c) & keep)[0])
        if kind == "control":
            a_rows.append(rows[:N_DEMO_LIKE])
            b_rows.append(rows[N_DEMO_LIKE:N_DEMO_LIKE + N_TRAIN_SIDE])
        else:
            b_rows.append(rows[:N_TRAIN_SIDE])
    B = F_pool.iloc[np.concatenate(b_rows)]
    A = F_pool.iloc[np.concatenate(a_rows)] if kind == "control" else (F_demo[~demo_flags] if peel else F_demo)
    return A, B

def run(kind, cols, K, reps, peel=False):
    res, oof = [], {}
    for k in range(K):
        A, B = draw(kind, k, peel)
        auc, s = adv_auc(A, B, cols, reps)
        res.append(auc)
        for i, v in zip(A.index, s):
            oof.setdefault(i, []).append(v)
    return np.array(res), {i: float(np.mean(v)) for i, v in oof.items()}

show = lambda name, r: print(f"  {name:<30} AUC {r.mean():.3f}   range [{r.min():.3f}, {r.max():.3f}]")

# ------------------------------------------------------------------ experiments
print("\nA. Control vs real (named measurements)")
ctrl, _ = run("control", NAMED, K=5, reps=3);        show("control (train vs train)", ctrl)
real, oof_demo = run("real", NAMED, K=5, reps=3);    show("REAL (demo vs train)", real)

print("\nB. Completeness gap: add the rich spectrum")
rich, _ = run("real", ALL, K=5, reps=3);             show("REAL, named + rich spectrum", rich)

print("\nC. Peeled: spike-flagged rows removed from both sides")
ctrl_p, _ = run("control", NAMED, K=5, reps=3, peel=True);  show("control (peeled)", ctrl_p)
real_p, _ = run("real", NAMED, K=5, reps=3, peel=True);     show("REAL (peeled)", real_p)

print(f"\nD. Group ablation on REAL (full named AUC = {real.mean():.3f})")
print(f"  {'group':<8} {'drop it':>8} {'only it':>8}")
for g in GROUPS:
    d, _ = run("real", [c for c in NAMED if group_of(c) != g], K=3, reps=2)
    o, _ = run("real", [c for c in NAMED if group_of(c) == g], K=3, reps=2)
    print(f"  {g:<8} {d.mean():>8.3f} {o.mean():>8.3f}")

print("\nE. Most demo-like demo rows (mean out-of-fold score, REAL run)")
print(f"  {'i':>4} {'score':>6} {'true':>5} {'spiked':>7} {'level':>7}")
for i in sorted(oof_demo, key=oof_demo.get, reverse=True)[:12]:
    print(f"  {i:>4} {oof_demo[i]:>6.3f} {int(test.labels[i]):>5} {str(bool(demo_flags[i])):>7} {float(test.meta['level'][i]):>7.2f}")

# Shift tests

In [ ]:
# CELL 0 — setup (run first). Needs Block 9a (demo_flags) and Block 11 (named_features, featurize, NAMED, group_of, F_demo).
import time, sys, importlib.util
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance

missing = [n for n in ["train", "test", "demo_flags", "named_features", "featurize", "NAMED", "group_of"] if n not in globals()]
assert not missing, f"run Block 9a / Block 11 first; missing {missing}"
B_PERM, ORIENT, CLIP, TRAIN_HOLDOUT_BASELINE = 3000, True, 10.0, False
GROUPS = ["shape", "nan", "hf", "mid", "lf", "spike", "scale", "xcorr", "meta"]
assert set(map(group_of, NAMED)) == set(GROUPS)
RES = []
t0 = time.time()

# baseline via Model(); batch path = same features + same classifier, much faster than 23 ms/call
spec = importlib.util.spec_from_file_location("baseline_model", "examples/baseline_model.py")
bm = importlib.util.module_from_spec(spec); sys.modules["baseline_model"] = bm; spec.loader.exec_module(bm)
base = bm.Model()
def base_feats(ds, idx): return np.stack([bm.features(ds[int(i)]) for i in idx])
def base_pred(ds, idx): return base.classifier.predict(base_feats(ds, idx))
chk = list(range(30))
assert all(int(base.predict(test[i])) == int(p) for i, p in zip(chk, base_pred(test, chk))), "batch path != Model.predict"

# splits and features
y_all = np.asarray(train.labels)
HOLD = np.sort(np.random.default_rng(0).permutation(len(train))[:20_000])      # README split
is_hold = np.zeros(len(train), bool); is_hold[HOLD] = True
r0 = np.random.default_rng(1)
TR_SIDE = np.sort(np.concatenate([r0.permutation(np.where((y_all == k) & ~is_hold)[0])[:250] for k in range(8)]))
F_trs = featurize(train, TR_SIDE)[NAMED]                                        # train side, 250/class, disjoint from HOLD
Xd = (F_demo if "F_demo" in globals() and len(F_demo) == len(test) else featurize(test, np.arange(len(test))))[NAMED]
y_d = np.asarray(test.labels)
cor_d = (base_pred(test, np.arange(len(test))) == y_d).astype(float)
acc_hold = (base_pred(train, HOLD) == y_all[HOLD]).mean()
print(f"baseline accuracy: demo {cor_d.mean():.3f} | HOLD 20k {acc_hold:.3f} (saved model was fit on all 100k)")

# helpers
def equal_bins(v, k, rng):
    v = np.asarray(v, float); n = len(v)
    order = np.lexsort((rng.random(n), v))              # random tie-break
    b = np.empty(n, int); b[order] = (np.arange(n) * k) // n
    return b

def delta_stat(bins, lo, hi):
    ml, mh = bins == lo, bins == hi
    return lambda c: c[ml].mean() - c[mh].mean()

def perm_p(stat_fn, correct, cls, B=B_PERM, seed=0):
    """Shuffle correctness within class. Returns (obs - null mean, two-sided p about the null mean)."""
    rng = np.random.default_rng(seed)
    groups = [np.where(cls == k)[0] for k in np.unique(cls)]
    obs, c, null = stat_fn(correct), correct.copy(), np.empty(B)
    for b in range(B):
        for g in groups:
            c[g] = correct[rng.permutation(g)]
        null[b] = stat_fn(c)
    mu = null.mean()
    return obs - mu, (1 + np.sum(np.abs(null - mu) >= abs(obs - mu) - 1e-12)) / (B + 1)

def new_clf(): return HistGradientBoostingClassifier(max_iter=100, learning_rate=0.08, max_depth=3, min_samples_leaf=20, random_state=0)

def adv_oof(Xpos, Xneg, reps=3, seed=0):
    X = np.vstack([Xpos, Xneg]); y = np.r_[np.ones(len(Xpos)), np.zeros(len(Xneg))].astype(int)
    w = compute_sample_weight("balanced", y); oof = np.zeros(len(X))
    for r in range(reps):
        for tr, va in StratifiedKFold(5, shuffle=True, random_state=seed + r).split(X, y):
            clf = new_clf().fit(X[tr], y[tr], sample_weight=w[tr])
            oof[va] += clf.predict_proba(X[va])[:, 1] / reps
    return oof[:len(Xpos)], roc_auc_score(y, oof)

def class_table(bins, nb, correct, cls, min_n=10):
    out = {f"Q{b+1}": {f"c{k}": (f"{correct[(bins==b)&(cls==k)].mean():.0%}" if ((bins==b)&(cls==k)).sum() >= min_n else "—")
                       + f"({((bins==b)&(cls==k)).sum()})" for k in range(8)} for b in range(nb)}
    return pd.DataFrame(out).T

pp = lambda x: f"{100*x:+.1f}"
print(f"setup done in {time.time()-t0:.0f}s")

# Error Analysis

In [ ]:
# BLOCK 7 (rewritten) — Shortcut check with honest cross-source evaluation.
# For each meta field:
#   (a) fit on TRAIN: value (or quantile-bin) -> majority class
#   (b) apply that train-derived map to DEMO: predicted = train_map[value]
#       report accuracy on demo (HONEST).
#   (c) also fit on DEMO itself and apply to demo (BIASED — uses demo labels
#       to build the map, upward-biased at n=80 per bin). Shown for comparison.
# Level binned into 5 quantile bins (edges from TRAIN percentiles).

def shortcut_check_honest(train, test, level_bins=5):
    print("=" * 82)
    print("SHORTCUT CHECK (honest) — train-derived majority map, applied to demo")
    print("=" * 82)

    def bin_values(arr, edges):
        if edges is None:
            return arr
        return np.digitize(arr.astype(float), edges)

    def fit_map(values, labels):
        """values -> majority class, on labeled rows only."""
        out = {}
        for v in np.unique(values):
            m = (values == v) & (labels >= 0)
            if m.sum() == 0:
                continue
            out[v] = int(np.bincount(labels[m], minlength=8).argmax())
        return out

    def apply_map(values, labels, mapping):
        """accuracy on rows with a label and a known mapping."""
        correct = total = 0
        for v, pred in mapping.items():
            m = (values == v) & (labels >= 0)
            if m.sum() == 0:
                continue
            correct += int((labels[m] == pred).sum())
            total   += int(m.sum())
        return (correct / total * 100) if total else 0.0, total

    for key in ["sample_rate", "group", "level"]:
        edges = None
        if key == "level":
            lv = train.meta["level"].astype(float)
            edges = np.percentile(lv, np.linspace(0, 100, level_bins + 1)[1:-1])

        tr_v = bin_values(train.meta[key], edges)
        te_v = bin_values(test.meta[key],  edges)

        # honest: map from train, applied to demo
        tr_map = fit_map(tr_v, train.labels)
        acc_honest, n_used = apply_map(te_v, test.labels, tr_map)

        # biased: map from demo, applied to demo
        te_map = fit_map(te_v, test.labels)
        acc_biased, _     = apply_map(te_v, test.labels, te_map)

        print(f"\n  [{key}]" + (f"  (binned into {level_bins} quantiles, edges from TRAIN)" if edges is not None else ""))
        print(f"    train-fitted map applied to demo (HONEST) : {acc_honest:5.2f}%  "
              f"({n_used} demo samples covered)")
        print(f"    demo-fitted map applied to demo (BIASED)  : {acc_biased:5.2f}%")
        print(f"    bias from fitting on demo's own labels    : {acc_biased - acc_honest:+5.2f} pp")
        print(f"    train bin -> majority class map: "
              + ", ".join(f"{k!s}={v}" for k, v in sorted(tr_map.items())))

shortcut_check_honest(train, test)

In [ ]:
# BLOCK 8 — Local scorer (exact match of harness.class_geomean_score) applied to
# the baseline's demo-test confusion matrix. Prints per-class recall and T_k.

import json, glob

def class_geomean_score(confusion, n_classes=8, wrong_penalty=0.33, alpha=0.5, p=2.0, eps=0.01):
    """confusion shape: (n_classes, n_classes + 2) — columns [0..K-1, abstain, failed]."""
    cm = np.asarray(confusion, dtype=float)
    K = n_classes
    n = np.maximum(cm.sum(axis=1), 1)          # total presented (incl abstain+failed)
    c = np.diag(cm[:, :K])                     # correct
    w = cm.sum(axis=1) - c - cm[:, K]          # wrong = not correct and not abstained
    t = n ** alpha * (c - wrong_penalty * w) / n * (c / n) ** p
    return float(np.exp(np.log(np.maximum(eps, t)).mean())), t

def baseline_score_card():
    files = sorted(glob.glob("results/baseline_model/run_*.json"))
    if not files:
        print("⚠️ No baseline results in results/baseline_model/")
        print("   Run: !python harness.py evaluate --model examples/baseline_model.py --source file:data/test_demo.npz")
        return
    res = json.load(open(files[0]))
    if "test_demo" not in str(res.get("source", "")):
        print(f"⚠️ Results are from {res.get('source')}, not demo test. Re-run on demo test.")
        return

    cm = np.array(res["confusion"])           # (8, 10)
    score, t = class_geomean_score(cm)
    n = cm.sum(axis=1)
    c = np.diag(cm[:, :8])
    recall = np.where(n > 0, c / n, 0.0)
    rank = {int(k): int(r + 1) for r, k in enumerate(np.argsort(t))}

    print("=" * 76)
    print(f"BASELINE SCORE CARD — {res['project']}, run {res['run']}, source={res['source']}")
    print(f"harness-reported score={res['score']:.4f}  accuracy={res['accuracy_all']:.4f}")
    print(f"local-recomputed score={score:.4f}   (should match harness)")
    print("=" * 76)
    print(f"  {'cls':>4} {'n':>5} {'correct':>8} {'recall':>8} {'T_k':>10} {'rank_by_T_k':>12}")
    for k in range(8):
        print(f"  {k:>4} {int(n[k]):>5} {int(c[k]):>8} {recall[k]:>7.1%} {t[k]:>10.4f} {rank[k]:>12}")
    print(f"\n  geometric mean     = {score:.4f}")
    print(f"  lowest-T_k class   = {int(np.argmin(t))}  (T_k={t.min():.4f})")
    print(f"  highest-T_k class  = {int(np.argmax(t))}  (T_k={t.max():.4f})")

baseline_score_card()

In [ ]:
# your submission, from a fresh clone, as we'll run it (write your repository's URL and remove the #)
# !git clone <your repository URL> /content/submission-check
# !pip install -q -r /content/submission-check/requirements.txt
# !python harness.py evaluate --model /content/submission-check/model.py